# Benchmark 360 - Extraccion Automatizada de Planes ISP

**Interact 2 Hack - Netlife / Ecuanet**

Este notebook ejecuta el pipeline de extraccion de datos de planes de internet
de ISPs competidores en Ecuador, comparando multiples modelos LLM y estrategias
de extraccion (HTML, OCR local, Vision LLM).

## Estructura
1. Setup y configuracion
2. Esquema de datos (30+ columnas)
3. Extraccion de un ISP de prueba
4. Comparacion multi-modelo (accuracy + costo)
5. Pipeline completo para todos los ISPs
6. Analisis de costos por MB
7. Visualizaciones

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

from settings import get_settings, LLM_PRICING, ISP_URLS, ISP_COMPANY_MAP
from schemas.plan import PlanISP
from schemas.cost_tracking import LLMCostRecord
from llm.cost_tracker import CostTracker

cfg = get_settings()
print(f'LLM Provider: {cfg.llm_provider}')
print(f'LLM Model: {cfg.llm_model}')
print(f'ISPs: {list(ISP_URLS.keys())}')
print(f'Eval models: {cfg.get_eval_models_list()}')

## 2. Esquema de Datos (30+ columnas)

El esquema `PlanISP` define las 30+ columnas requeridas por el reto.

In [ ]:
import json

schema = PlanISP.model_json_schema()
props = schema.get('properties', {})

print(f'Total campos: {len(props)}')
print()

# Tabla de campos
fields_df = pd.DataFrame([
    {
        'campo': name,
        'tipo': str(info.get('type', info.get('anyOf', 'complex'))),
        'descripcion': info.get('description', '')[:60],
    }
    for name, info in props.items()
])
fields_df

## 3. Tabla de Costos por Modelo LLM

Precios de referencia para analisis de imagenes con cada modelo.

In [ ]:
pricing_df = pd.DataFrame([
    {
        'modelo': model,
        'input_usd_1M_tokens': p['input'],
        'output_usd_1M_tokens': p['output'],
        'costo_estimado_1MB_imagen': round(
            (1600 / 1_000_000) * p['input'] + (800 / 1_000_000) * p['output'], 6
        ),
        'gratis': p['input'] == 0,
    }
    for model, p in LLM_PRICING.items()
])

pricing_df = pricing_df.sort_values('costo_estimado_1MB_imagen', ascending=False)
print('Tabla de Precios por Modelo LLM')
print('=' * 80)
pricing_df

## 4. Extraccion de un ISP de Prueba

Probamos la extraccion con un solo ISP para validar el pipeline.

In [ ]:
from pipeline.runner import run_single_isp
from pipeline.parquet_writer import plans_to_dataframe

# Extraer de un ISP (cambiar segun necesidad)
test_isp = cfg.eval_isp  # default: 'xtrim'
print(f'Extrayendo de: {test_isp}')

# Usa strategy='html' para probar sin LLM (gratis)
# Usa strategy='llm' para usar Vision LLM
plans = run_single_isp(test_isp, strategy='llm')

print(f'\nPlanes extraidos: {len(plans)}')
if plans:
    df = plans_to_dataframe(plans)
    df[['nombre_plan', 'velocidad_download_mbps', 'precio_plan', 'tecnologia']]

## 5. Evaluacion Multi-Modelo

Comparamos todos los modelos configurados sobre la misma pagina ISP.
Esto genera la tabla clave de **costo por MB** y **accuracy**.

In [ ]:
from pipeline.evaluator import evaluate_models

# Ejecutar evaluacion (requiere API keys configuradas)
eval_df = evaluate_models(isp_key=test_isp)

print('\n=== Comparacion Multi-Modelo ===')
print(f'ISP evaluado: {test_isp}')
print()
eval_df

## 6. Analisis de Costos por MB

In [ ]:
tracker = CostTracker()
cost_summary = tracker.summary()

if not cost_summary.empty:
    print('=== Resumen de Costos ===')
    print()
    display(cost_summary)
    
    print()
    print('Detalle por llamada:')
    cost_df = tracker.to_dataframe()
    display(cost_df[[
        'model', 'isp', 'image_size_mb', 'cost_usd',
        'cost_per_mb', 'latency_ms', 'field_coverage_pct', 'plans_extracted'
    ]])
else:
    print('No hay datos de costo. Ejecuta la seccion 4 o 5 primero.')

## 7. Pipeline Completo (Todos los ISPs)

In [ ]:
from pipeline.runner import run_all_isps
from pipeline.parquet_writer import write_parquet

# Extraer de todos los ISPs
# NOTA: Esto usa el modelo LLM configurado y consume API credits
all_plans = run_all_isps(strategy='llm')

print(f'\nTotal planes extraidos: {len(all_plans)}')

if all_plans:
    output_path = write_parquet(all_plans)
    print(f'Parquet guardado: {output_path}')
    
    # Exportar costos
    tracker.export_parquet()
    
    df_all = plans_to_dataframe(all_plans)
    print(f'\nResumen por ISP:')
    display(df_all.groupby('marca').agg(
        planes=('nombre_plan', 'count'),
        precio_min=('precio_plan', 'min'),
        precio_max=('precio_plan', 'max'),
        vel_max=('velocidad_download_mbps', 'max'),
    ).round(2))

## 8. Visualizaciones

In [ ]:
if 'df_all' in dir() and not df_all.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Precio vs Velocidad por ISP
    for marca in df_all['marca'].unique():
        subset = df_all[df_all['marca'] == marca]
        axes[0].scatter(
            subset['velocidad_download_mbps'],
            subset['precio_plan'],
            label=marca, s=80, alpha=0.7,
        )
    axes[0].set_xlabel('Velocidad Download (Mbps)')
    axes[0].set_ylabel('Precio Plan (USD sin IVA)')
    axes[0].set_title('Precio vs Velocidad por ISP')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Costo por MB por modelo
    if not cost_summary.empty:
        cost_summary_sorted = cost_summary.sort_values('avg_cost_per_mb')
        colors = ['green' if c == 0 else 'steelblue' 
                  for c in cost_summary_sorted['avg_cost_per_mb']]
        axes[1].barh(
            cost_summary_sorted['model'],
            cost_summary_sorted['avg_cost_per_mb'],
            color=colors,
        )
        axes[1].set_xlabel('Costo Promedio por MB (USD)')
        axes[1].set_title('Costo por MB por Modelo LLM')
        axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('../data/processed/benchmark_visualizations.png', dpi=150)
    plt.show()
else:
    print('Ejecuta la seccion 7 primero para generar visualizaciones.')

## 9. Verificar Parquet de Salida

In [ ]:
parquet_path = Path('../data/processed/benchmark_industria.parquet')

if parquet_path.exists():
    df_check = pd.read_parquet(parquet_path)
    print(f'Registros: {len(df_check)}')
    print(f'Columnas: {len(df_check.columns)}')
    print(f'ISPs: {df_check["marca"].unique().tolist()}')
    print()
    print('Primeros registros:')
    display(df_check.head())
    
    print(f'\nColumnas ({len(df_check.columns)}):')
    for col in df_check.columns:
        print(f'  - {col}: {df_check[col].dtype}')
else:
    print('Parquet no existe. Ejecuta la seccion 7 primero.')